# Check dataset columns

In [1]:

import os
# Change directory to import properly
os.chdir('..')

from src.data_loader import load_besstie, get_train_conditions, get_test_conditions

# Load besstie dataset
ds = load_besstie()

# Look at the format of a besttie entry
print(ds['train'][0])

c:\Users\joela\Documents\code\ai\nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'text': "I'm a member of the Green Party but I'll be voting Lib Dem as it's so tight here between Lib Dem and Tory. I cannot contemplate our useless tit of a Tory MP being reelected. I'll use the Swap My Vote website so someone somewhere can vote Green for me.", 'variety': 'en-UK', 'source': 'Reddit', 'Sentiment': 0.0, 'Sarcasm': 0.0}


In [7]:
from datasets import Dataset

from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
)

import torch 
import numpy as np

SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

TASK      = "sarcasm"
LABEL_COL = "Sarcasm"
TEXT_COL  = "text"

MODEL_NAME = "roberta-base"
tokenizer  = RobertaTokenizer.from_pretrained(MODEL_NAME)
TASK = "sarcasm"

# Function to tokenize each bacth of the dataset
def tokenize(batch):
    tokens = tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = batch[LABEL_COL]

    return tokens

def prepare_dataset(dataset):
    """Tokenizes and formats dataset for trainer"""

    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.remove_columns(
        [c for c in tokenized.column_names if c not in
        ["input_ids", "attention_mask", "labels"]]
    )
    
    tokenized.set_format("torch")
    return tokenized

In [8]:
# Functions to compute model accuracy
from sklearn.metrics import (
    f1_score, precision_score,
    recall_score, confusion_matrix,
    classification_report
)

def compute_metrics(eval_pred):
    """Returns macro-F1."""
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "accuracy": (predictions == labels).mean()
    }

def full_evaluation(y_true, y_pred):
    """Full metrics for report — called after training completes."""
    return {
        "macro_f1":        round(f1_score(y_true, y_pred, average="macro"), 4),
        "precision":       round(precision_score(y_true, y_pred, average="macro"), 4),
        "recall":          round(recall_score(y_true, y_pred, average="macro"), 4),
        "per_class_f1":    f1_score(y_true, y_pred, average=None).tolist(),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "report":          classification_report(
                                y_true, y_pred,
                                target_names=["Not Sarcastic", "Sarcastic"])
    }

In [9]:
# Training function - using trainer instead of manual loop
def train_roberta(train_data, val_data, seed=42, output_dir="./tmp"):
    """Fine tunes RoBERTa on given training data. Returns 
    trained model and tokeizer"""

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    # Tokenize training data and validations values
    train_tokenized = prepare_dataset(train_data)
    val_tokenized = prepare_dataset(val_data)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return model, tokenizer

def evaluate_on_testset(model, test_data):
    """Runs trained model on a test set. Returns 
        evaluation metric dict """
    
    test_tokenized = prepare_dataset(test_data)

    trainer = Trainer(model=model)
    output = trainer.predict(test_tokenized)

    y_pred = np.argmax(output.predictions, axis=1)
    y_true = test_tokenized["labels"].numpy()

    return full_evaluation(y_true, y_pred)

In [10]:
print("Train columns:", ds["train"].column_names)

Train columns: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm']


In [11]:
# Experiment loop
from tqdm import tqdm
import json

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

all_results = {}
SEEDS = [42, 123]

train_conditions = get_train_conditions(ds)
test_sets = get_test_conditions(ds)
val_data = ds["validation"]

# Training conditions
for condition_name, train_data in tqdm(
    train_conditions.items(),
    desc="Conditions",
    position=0
):
    condition_results = {}

    # Loop for the seeds
    for seed in tqdm(
        SEEDS,
        desc=f" {condition_name}",
        position=1,
        leave=False
    ):
        model, tok = train_roberta(
            train_data, val_data,
            seed=seed,
            output_dir=f"./tmp/{condition_name}_seed{seed}"
        )

        seed_results = {}
        for test_name, test_data in tqdm(
            test_sets.items(),
            desc=f" Evaluating",
            position=2,
            leave=False
        ):
            results = evaluate_on_testset(model, test_data)
            seed_results[test_name] = results

        condition_results[f"seed_{seed}"] = seed_results

    averaged = {}

    for test_name in test_sets.keys():
        f1_scores = [
            condition_results[f"seed_{s}"][test_name]["macro_f1"]
            for s in SEEDS
        ]
        averaged[test_name] = {
            "macro_f1_mean": round(np.mean(f1_scores), 4),
            "macro_f1_std": round(np.std(f1_scores), 4),
        }

    all_results[condition_name] = {
        "by_seed": condition_results,
        "averaged": averaged
    }
    with open(f"results/{condition_name}.json", "w") as f:
        json.dump(all_results[condition_name], f, indent=2)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3783.48it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map: 100%|██████████| 1203/1203 [00:00<00

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Results Visualisation (incl cross variety matrix)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

conditions = list(train_conditions.keys())
test_names = list(test_sets.keys())

matrix = np.array([
    [all_results[c]]["averaged"][t]["macro_f1_mean"]
    for t in test_names]
    for c in conditions
)

plt.figure(figsize=(8,6))

sns.heatmap(
    matrix,
    annot=True,
    fmt='.3f',
    xticklabels=["Test UK", "Test AU", "Test IN"],
    yticklabels=["UK only", "AU only", "IN only",
                 "Inner pool", "All pool"],
    cmap="YlOrRd",
    vmin=0.5, vmax=1.0
)
plt.title("Cross-Variety Evaluation Matrix — Macro-F1 (RoBERTa)")
plt.ylabel("Trained on")
plt.xlabel("Tested on")
plt.tight_layout()
plt.savefig("figures/cross_variety_matrix.png", dpi=150)
plt.show()

results_df = pd.DataFrame(
    matrix,
    index=["UK only", "AU only", "IN only", "Inner pool", "All pool"],
    columns=["Test UK", "Test AU", "Test IN"]
)
print("\nCross-Variety Matrix:")
print(results_df.to_string())

In [ ]:
# Find confusion matrix for best conditions
best_condition = max(
    conditions,
    key=lambda c: np.mean([
        all_results[c]["averaged"][t]["macro_f1_mean"]
        for t in test_names
    ])
)
print(f"Best condition: {best_condition}")

best_cm = np.array(
    all_results[best_condition]["by_seed"]["seed_42"]["uk_test"]["confusion_matrix"]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    best_cm,
    annot=True,
    fmt="d",
    xticklabels=["Not Sarcastic", "Sarcastic"],
    yticklabels=["Not Sarcastic", "Sarcastic"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_condition} → UK test")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("figures/confusion_matrix_best.png", dpi=150)
plt.show()